In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [38]:
# Step 1: Get all product links from the shop page
base_url = "https://scrapeme.live/shop/page/{}/"
products_links = []

for page in range(1, 49):
    response = requests.get(base_url.format(page), verify=False)
    soup = BeautifulSoup(response.text, "html.parser")
    product_elements = soup.find_all("h2", class_="woocommerce-loop-product__title")

    for product in product_elements:
        link = product.find_parent("a")["href"]
        products_links.append(link)

In [40]:
product_elements

[<h2 class="woocommerce-loop-product__title">Naganadel</h2>,
 <h2 class="woocommerce-loop-product__title">Stakataka</h2>,
 <h2 class="woocommerce-loop-product__title">Blacephalon</h2>]

In [41]:
products_links

['https://scrapeme.live/shop/Bulbasaur/',
 'https://scrapeme.live/shop/Ivysaur/',
 'https://scrapeme.live/shop/Venusaur/',
 'https://scrapeme.live/shop/Charmander/',
 'https://scrapeme.live/shop/Charmeleon/',
 'https://scrapeme.live/shop/Charizard/',
 'https://scrapeme.live/shop/Squirtle/',
 'https://scrapeme.live/shop/Wartortle/',
 'https://scrapeme.live/shop/Blastoise/',
 'https://scrapeme.live/shop/Caterpie/',
 'https://scrapeme.live/shop/Metapod/',
 'https://scrapeme.live/shop/Butterfree/',
 'https://scrapeme.live/shop/Weedle/',
 'https://scrapeme.live/shop/Kakuna/',
 'https://scrapeme.live/shop/Beedrill/',
 'https://scrapeme.live/shop/Pidgey/',
 'https://scrapeme.live/shop/Pidgeotto/',
 'https://scrapeme.live/shop/Pidgeot/',
 'https://scrapeme.live/shop/Rattata/',
 'https://scrapeme.live/shop/Raticate/',
 'https://scrapeme.live/shop/Spearow/',
 'https://scrapeme.live/shop/Fearow/',
 'https://scrapeme.live/shop/Ekans/',
 'https://scrapeme.live/shop/Arbok/',
 'https://scrapeme.live/

In [42]:
# Step 2: Extract details from each product page
product_name = []
product_price = []
product_description = []
product_no_InStock = []
product_categories = []
product_tags = []

for link in products_links:
    response = requests.get(link, verify=False)
    soup = BeautifulSoup(response.text, "html.parser")

    # Name
    name = soup.find("h1", class_="product_title entry-title").text
    # Price
    price = soup.find("p", class_="price").text
    # Description
    description_element = soup.find("div", class_="woocommerce-product-details__short-description")
    description = description_element.text.strip() if description_element else "No description available"
    # Stock
    stock = soup.find("p", class_="stock in-stock").text
    # Categories
    category_span = soup.find("span", class_="posted_in")
    categories = [cat.text for cat in category_span.find_all("a")] if category_span else []
    # Tags
    tags_span = soup.find("span", class_="tagged_as")
    tags = [tag.text for tag in tags_span.find_all("a")] if tags_span else []

    # Append
    product_name.append(name)
    product_price.append(price)
    product_description.append(description)
    product_no_InStock.append(stock)
    product_categories.append(categories)
    product_tags.append(tags)


In [44]:
print(product_name)
print(product_price)
print(product_description)
print(product_no_InStock)
print(product_categories)
print(product_tags)

['Bulbasaur', 'Ivysaur', 'Venusaur', 'Charmander', 'Charmeleon', 'Charizard', 'Squirtle', 'Wartortle', 'Blastoise', 'Caterpie', 'Metapod', 'Butterfree', 'Weedle', 'Kakuna', 'Beedrill', 'Pidgey', 'Pidgeotto', 'Pidgeot', 'Rattata', 'Raticate', 'Spearow', 'Fearow', 'Ekans', 'Arbok', 'Pikachu', 'Raichu', 'Sandshrew', 'Sandslash', 'Nidorina', 'Nidoqueen', 'Nidorino', 'Nidoking', 'Clefairy', 'Clefable', 'Vulpix', 'Ninetales', 'Jigglypuff', 'Wigglytuff', 'Zubat', 'Golbat', 'Oddish', 'Gloom', 'Vileplume', 'Paras', 'Parasect', 'Venonat', 'Venomoth', 'Diglett', 'Dugtrio', 'Meowth', 'Persian', 'Psyduck', 'Golduck', 'Mankey', 'Primeape', 'Growlithe', 'Arcanine', 'Poliwag', 'Poliwhirl', 'Poliwrath', 'Abra', 'Kadabra', 'Alakazam', 'Machop', 'Machoke', 'Machamp', 'Bellsprout', 'Weepinbell', 'Victreebel', 'Tentacool', 'Tentacruel', 'Geodude', 'Graveler', 'Golem', 'Ponyta', 'Rapidash', 'Slowpoke', 'Slowbro', 'Magnemite', 'Magneton', 'Farfetchd', 'Doduo', 'Dodrio', 'Seel', 'Dewgong', 'Grimer', 'Muk', 'S

In [45]:
# Step 3: Create DataFrame
df = pd.DataFrame({
    "Name": product_name,
    "Price": product_price,
    "Description": product_description,
    "Stock": product_no_InStock,
    "Categories": product_categories,
    "Tags": product_tags
})

# Save to CSV for EDA
df.to_csv("final_products.csv", index=False)
print("Scraping Done ✅ Data saved to products.csv")

Scraping Done ✅ Data saved to products.csv


In [48]:
# Display descriptive statistics of the DataFrame
display(df.describe())

,Price_numeric
count,755.000000
mean,110.948344
std,51.702359
min,25.000000
25%,66.000000
50%,111.000000
75%,157.000000
max,200.000000


In [49]:
import plotly.express as px

# Convert price to numeric
df['Price_numeric'] = df['Price'].str.replace('£', '').astype(float)

# Histogram of Product Prices
fig_price = px.histogram(df, x='Price_numeric', title='Distribution of Product Prices')
fig_price.show()

# Bar chart of top Categories
# Filter out 'Pokemon' and extract the other category
def extract_other_category(categories_list):
    if isinstance(categories_list, list):
        other_categories = [cat for cat in categories_list if cat != 'Pokemon']
        return other_categories[0] if other_categories else None
    return None

df['Other_Category'] = df['Categories'].apply(extract_other_category)

# Count the occurrences of the extracted categories
other_category_counts = df['Other_Category'].value_counts().reset_index()
other_category_counts.columns = ['Category', 'Count']

fig_categories = px.bar(other_category_counts.head(10), x='Category', y='Count', title='Top 10 Product Categories (excluding Pokemon)')
fig_categories.show()

In [47]:
import plotly.express as px

# Box plot of Product Prices to show the range and distribution
fig_box = px.box(df, y='Price_numeric', title='Product Price Range')
fig_box.show()